# Extracción de muestras de evidencia conservando las IPGenera los CSV de muestra que se suben a la pestaña **⚖️ Formal** de MODEXRE,a partir de los CSV limpios completos (`*_full_clean.csv`), **manteniendo todaslas columnas del original**: `source_ip`/`src`, `destination_ip`/`dst`, lospuertos y la marca temporal.## Por qué hace faltaLas muestras generadas con los cuadernos de limpieza antiguos descartaban IP,puerto y marca temporal. Sin esas columnas:- el informe pericial muestra `IPs de origen: n/d`, porque no hay direcciones  que mostrar;- y las cuatro características de agregación (`agg_distinct_dst_hosts`,  `agg_distinct_dst_ports`, `agg_events_in_window`, `agg_events_per_second`)  salen a cero, porque se calculan agrupando por IP de origen y ordenando por  tiempo.## Por qué el muestreo no es aleatorioEsas cuatro características miden **densidad temporal por origen**: cuántosdestinos y puertos distintos toca una misma IP dentro de una ventana de 60 s.Un muestreo aleatorio estratificado reparte las filas de un barrido entre milesde huecos y destruye esa densidad: de un escaneo real de 1.000 puertos tequedas con 70 sueltos, y lo que llega al clasificador ya no se parece a unbarrido. Por eso el modo por defecto es **`coherente`**, que conserva secuenciasconsecutivas del mismo origen en orden temporal.

In [ ]:
# Dependencias (en Colab ya suelen estar)
!pip install -q pandas tqdm

import os
from collections import defaultdict

import pandas as pd
from tqdm.auto import tqdm

print("pandas", pd.__version__)

## 1. Origen de los ficherosElige **una** de las dos celdas siguientes.> No uses `files.upload()`: para un CSV de varios GB la transferencia por el> navegador puede tardar horas. Montar Drive o leer de una ruta local es> inmediato.

In [ ]:
# --- OPCIÓN A: Google Drive (recomendado en Colab) ---
from google.colab import drive
drive.mount('/content/drive')

# Las rutas de FICHEROS (paso 2) son absolutas, así que CARPETA
# solo actúa de prefijo si alguna fuera relativa.
CARPETA = ''

In [ ]:
# --- OPCIÓN B: ruta local (Jupyter en tu equipo) ---
# CARPETA = r'C:\Proyectos\Datasets\Dataset_05082026'

## 2. Configuración`FILAS_POR_CLASE` es el parámetro que más importa. Con 71 filas por clase —comoen las muestras antiguas— un barrido se queda por debajo del umbral que`model_v3` aprendió y acaba clasificado como `Normal`. **Usa 500 o más.**

In [ ]:
# Rutas reales de los CSV limpios COMPLETOS, los que conservan la IP.
# Verificadas: CICIDS trae source_ip/destination_ip/timestamp, UNSW
# trae srcip/dstip/stime, y Kitsune src/dst/time.
#
# Las versiones sin sufijo _v2 de MasterIA/Proyectos/Datasets son las
# de la limpieza ANTIGUA y NO llevan IP: no usarlas aquí.
FICHEROS = {
    'cicids2017': ('/content/drive/MyDrive/Datasets/Dataset06082026/MachineLearningCVE_full_clean_v2.csv',
                   'cicids2017_muestra_pequena.csv'),
    'unsw_nb15':  ('/content/drive/MyDrive/Datasets/Dataset06082026/UNSW_NB15_full_clean_v2.csv',
                   'unsw_nb15_muestra_pequena.csv'),
    'kitsune':    ('/content/drive/MyDrive/MasterIA/Proyectos/Datasets/KITSUNE_full_clean.csv',
                   'kitsune_muestra_pequena.csv'),
}

FILAS_POR_CLASE = 500      # con menos de ~200 el barrido queda diluido
MODO            = 'coherente'   # 'coherente' | 'aleatorio'
SALIDA          = '/content'
SEMILLA         = 42

TAMANO_BLOQUE  = 250_000
FACTOR_RESERVA = 30

## 3. FuncionesLos alias de columna son **los mismos** que usa `app/ocsf/mappers.py` enMODEXRE, así que lo que genere este cuaderno es exactamente lo que el sistemasabe leer: `src`/`dst`/`time` en Kitsune, `source_ip`/`destination_ip`/`timestamp` en CICIDS2017, `srcip`/`dstip`/`stime` en UNSW-NB15.

In [ ]:
ALIAS_SRC  = ('src_ip', 'source_ip', 'srcip', 'saddr', 'ip_src', 'source_address', 'src')
ALIAS_DST  = ('dst_ip', 'destination_ip', 'dstip', 'daddr', 'ip_dst', 'destination_address', 'dst')
ALIAS_TIME = ('time', 'timestamp', 'stime', 'flow_start', 'first_seen', 'ts')


def primera(columnas, alias):
    """Primer alias presente entre las columnas, o None."""
    for nombre in alias:
        if nombre in columnas:
            return nombre
    return None


def normalizar_columnas(cols):
    """Misma normalización que los cuadernos FULL_* y la capa de ingesta de
    MODEXRE: strip, espacios/guiones/barras a '_', minúsculas."""
    return [str(c).strip().replace(' ', '_').replace('/', '_').replace('-', '_').lower()
            for c in cols]

In [ ]:
def extraer(entrada, salida, por_clase=FILAS_POR_CLASE, modo=MODO, semilla=SEMILLA):
    """Extrae una muestra por clase conservando todas las columnas."""
    print(f'>> {os.path.basename(entrada)}')
    if not os.path.exists(entrada):
        print('   NO ENCONTRADO - revisa CARPETA y el nombre del fichero\n')
        return None

    reserva = defaultdict(list)
    col_src = col_dst = col_time = None
    tope = por_clase * FACTOR_RESERVA
    total = 0

    with tqdm(desc='   leyendo', unit=' filas', unit_scale=True) as barra:
        for bloque in pd.read_csv(entrada, chunksize=TAMANO_BLOQUE, low_memory=False):
            bloque.columns = normalizar_columnas(bloque.columns)

            if col_src is None and 'attack_cat' not in bloque.columns:
                raise ValueError("El CSV no tiene columna 'attack_cat'. "
                                 '¿Es uno de los *_full_clean.csv?')
            if total == 0:
                cols = set(bloque.columns)
                col_src  = primera(cols, ALIAS_SRC)
                col_dst  = primera(cols, ALIAS_DST)
                col_time = primera(cols, ALIAS_TIME)

            total += len(bloque)
            barra.update(len(bloque))
            for clase, filas in bloque.groupby('attack_cat', observed=True):
                if sum(len(x) for x in reserva[clase]) < tope:
                    reserva[clase].append(filas)

    print(f'   columnas -> origen: {col_src or "NINGUNA"} | '
          f'destino: {col_dst or "NINGUNA"} | tiempo: {col_time or "NINGUNA"}')
    if col_src is None:
        print('   AVISO: este fichero NO tiene columna de IP de origen.')
        print('          La muestra saldrá sin direcciones y MODEXRE mostrará "n/d".')
        print('          Probablemente has apuntado al CSV limpio ANTIGUO.')

    partes = []
    for clase in sorted(reserva):
        datos = pd.concat(reserva[clase], ignore_index=True)

        if modo == 'coherente' and col_src and col_time:
            datos = datos.sort_values(col_time)
            # Origen con más actividad: el actor que concentra el patrón
            # (el que barre, el que inunda...). Conservar SUS eventos
            # consecutivos es lo que mantiene viva la señal de agregación.
            principal = datos[col_src].value_counts().idxmax()
            del_principal = datos[datos[col_src] == principal].head(por_clase)
            resto = por_clase - len(del_principal)
            if resto > 0:
                otros = datos[datos[col_src] != principal].head(resto)
                elegidas = pd.concat([del_principal, otros], ignore_index=True)
            else:
                elegidas = del_principal
            detalle = f'origen principal {principal} aporta {len(del_principal)}'
        else:
            elegidas = datos.sample(n=min(por_clase, len(datos)), random_state=semilla)
            detalle = 'muestreo aleatorio'

        partes.append(elegidas)
        print(f'   {clase:<28} {len(elegidas):>5} filas   ({detalle})')

    muestra = pd.concat(partes, ignore_index=True)
    if col_time:
        muestra = muestra.sort_values(col_time).reset_index(drop=True)
    muestra.to_csv(salida, index=False)

    print(f'   GUARDADO {salida}  ->  {len(muestra):,} filas, '
          f'{len(muestra.columns)} columnas', end='')
    print(f', {muestra[col_src].nunique()} IPs de origen\n' if col_src else '\n')
    return muestra

## 4. Generar las muestras

In [ ]:
muestras = {}
for clave, (nombre_entrada, nombre_salida) in FICHEROS.items():
    muestras[clave] = extraer(
        os.path.join(CARPETA, nombre_entrada),
        os.path.join(SALIDA, nombre_salida),
    )

## 5. ComprobaciónVerifica que las muestras llevan dentro lo que MODEXRE necesita. Si unacolumna de origen sale como `NINGUNA`, esa muestra seguirá mostrando `n/d`en el informe.

In [ ]:
for clave, df in muestras.items():
    if df is None:
        continue
    cols = set(df.columns)
    src, dst, tiempo = (primera(cols, a) for a in (ALIAS_SRC, ALIAS_DST, ALIAS_TIME))
    print(f'{clave}:')
    print(f'   {len(df):,} filas | {df["attack_cat"].nunique()} clases')
    print(f'   origen: {src or "NINGUNA"} | destino: {dst or "NINGUNA"} | tiempo: {tiempo or "NINGUNA"}')
    if src:
        print(f'   {df[src].nunique()} IPs de origen distintas')
        top = df[src].value_counts().head(3)
        print('   más activas: ' + ', '.join(f'{ip} ({n})' for ip, n in top.items()))
    print()

In [ ]:
# Descargar las muestras al equipo (solo en Colab)
from google.colab import files
for _, nombre_salida in FICHEROS.values():
    ruta = os.path.join(SALIDA, nombre_salida)
    if os.path.exists(ruta):
        files.download(ruta)

## 6. Qué hacer con las muestras1. Abre MODEXRE y ve a la pestaña **⚖️ Formal**.2. Sube una de las muestras generadas como fichero de evidencia.3. Pulsa **Ejecutar análisis pericial**.En el **Resumen por tipo de ataque** deberían aparecer ya las direcciones dondeantes ponía `IPs de origen: n/d`.---**Requisito:** MODEXRE tiene que llevar aplicado el parche de`backend/app/ocsf/mappers.py` que emite `src_endpoint`/`dst_endpoint`/`time`.Para comprobarlo, desde la carpeta del proyecto:```python -c "from pathlib import Path; print('OK' if '_SRC_IP_ALIASES' in Path('backend/app/ocsf/mappers.py').read_text(encoding='utf-8') else 'FALTA EL PARCHE')"```